# Geração de Amostras V7 — Fase 2 (Alta Performance + Checkpoint)

Este notebook executa o enriquecimento dos arquivos NPZ gerados na Fase 1, calculando os *soft-targets* (melhor jogada e scores) via **Minimax com profundidade 11**.

### Otimizações e Segurança:
1. **Databricks Serverless Compatible:** Utiliza a API `mapInPandas` ao invés de RDDs, evitando o erro `JVM_ATTRIBUTE_NOT_SUPPORTED` em Shared Clusters.
2. **Checkpointing (Lotes de 20k):** Processa e grava o resultado de 4 em 4 arquivos NPZ (~20.000 amostras). Se o cluster cair, o trabalho é retomado automaticamente a partir do último lote não processado.
3. **Gravação Atômica:** Utiliza arquivos temporários (`.tmp.npz`) para garantir que uma queda de energia ou travamento durante a gravação não corrompa os dados já salvos.
4. **Bitboard & Transposition Table:** As simulações do Minimax utilizam álgebra booleana de 64 bits e cache de transposição para velocidade extrema.

**Diretório de Trabalho:** `/Workspace/Users/diondu@gmail.com/CNN/profundidade_11_v7_adaptativo`

> **⚠️ ALERTA DE MANUTENÇÃO: BUGS ALGÓRITMICOS NO BITBOARD (Maio/2026)**
>
> Este algoritmo Minimax via Bitboard possui duas particularidades matemáticas gravíssimas que foram corrigidas na versão atual, mas que **não devem ser alteradas ou esquecidas** caso esse código seja adaptado para outros formatos:
> 
> 1. **Caixas Pré-Fechadas (Falsos Positivos):** Como a matriz vira bits, adicionar um traço não pode contar simplesmente se a máscara da caixa ficou completa `(edges | bit) & mask == mask`. É OBRIGATÓRIO checar se a caixa *não estava fechada antes* `and (edges & mask) != mask`. Sem isso, o Bitboard injeta pontos fantasmas durante a simulação em profundidade e corrompe o score.
> 2. **Offsets na Poda Alpha-Beta Incremental:** Diferente de motores comuns, este Bitboard retorna *scores incrementais* em vez de absolutos. Por causa disso, ao chamar a subárvore no `solve_minimax_bitboard`, nós **temos que repassar os limites `alpha` e `beta` com um offset de `- closed` (ou `+ closed` pro adversário)**. Se você repassar o `alpha` puro, a subárvore sofre "poda prematura" e o motor escolhe jogadas perdedoras por achar que metas inalcançáveis não valem a pena serem processadas.

In [0]:
import os
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path

# Configurações
INPUT_DIR = Path("/Workspace/Users/diondu@gmail.com/CNN/profundidade_11_v7_adaptativo")
DEPTH_TARGET = 11
LOTE_ARQUIVOS = 4  # Processa 4 NPZs por vez (~20.000 amostras) para garantir checkpoints

print(f"Iniciando monitoramento da pasta: {INPUT_DIR}")

Iniciando monitoramento da pasta: /Workspace/Users/diondu@gmail.com/CNN/profundidade_11_v7_adaptativo


In [0]:
def processar_lote_pandas(iterator):
    """
    Função enviada aos Workers do PySpark.
    Toda a lógica de Bitboard está envelopada aqui dentro para evitar 
    erros de serialização (Pickle) no Databricks Serverless.
    """
    import pandas as pd
    import numpy as np
    import random
    
    # 1. Reconstrói as tabelas de Bitboard localmente no Worker
    h, w = 9, 7
    edge_to_bit = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                edge_to_bit[(r, c)] = bit_idx
                bit_to_label[bit_idx] = f"H_{r}_{c}" if r % 2 == 0 else f"V_{r}_{c}"
                bit_idx += 1
                
    n_edges = bit_idx
    all_mask = (1 << n_edges) - 1
    
    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            mask = (1 << edge_to_bit[(r-1, c)]) | (1 << edge_to_bit[(r+1, c)]) | \
                   (1 << edge_to_bit[(r, c-1)]) | (1 << edge_to_bit[(r, c+1)])
            box_masks.append(mask)
            
    edge_boxes = [tuple(bm for bm in box_masks if bm & (1 << b)) for b in range(n_edges)]
    
    # 2. Definição do Minimax Otimizado (Fix Caixas + Offset Alpha-Beta + TT Completa)
    def solve_minimax_bitboard(edges, depth, alpha, beta, maximizing, tt):
        if depth == 0 or edges == all_mask: return 0
        
        tt_key = (edges, depth, maximizing)
        if tt_key in tt:
            flag, val = tt[tt_key]
            if flag == 0: return val
            if flag == 1 and val >= beta: return val
            if flag == 2 and val <= alpha: return val

        moves = []
        for i in range(n_edges):
            if not (edges & (1 << i)):
                ne = edges | (1 << i)
                # Conta apenas se completou os 4 lados AGORA (nao conta pre-fechadas)
                closed = sum(1 for bm in edge_boxes[i] if ne & bm == bm and edges & bm != bm)
                moves.append((i, closed))
        moves.sort(key=lambda x: x[1], reverse=True)
        
        orig_alpha = alpha
        orig_beta = beta
        best_val = -10000 if maximizing else 10000
        
        for bit, closed in moves:
            new_e = edges | (1 << bit)
            if maximizing:
                val = (closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)) if closed > 0 else \
                      solve_minimax_bitboard(new_e, depth-1, alpha, beta, False, tt)
                best_val = max(best_val, val)
                alpha = max(alpha, best_val)
            else:
                val = (-closed + solve_minimax_bitboard(new_e, depth-1, alpha + closed, beta + closed, False, tt)) if closed > 0 else \
                      solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
                best_val = min(best_val, val)
                beta = min(beta, best_val)
            if beta <= alpha: break

        # RESTAURAÇÃO DA TT: Gravando Bounds (Limites)
        if maximizing:
            if best_val <= orig_alpha:
                flag = 2 # UPPERBOUND
            elif best_val >= beta:
                flag = 1 # LOWERBOUND
            else:
                flag = 0 # EXACT
        else:
            if best_val >= orig_beta:
                flag = 1 # LOWERBOUND
            elif best_val <= alpha:
                flag = 2 # UPPERBOUND
            else:
                flag = 0 # EXACT

        tt[tt_key] = (flag, best_val)
        return best_val

    # 3. Processamento do DataFrame do Spark
    for pdf in iterator:
        resultados = []
        for estado_bytes in pdf['estado_bytes']:
            # Conversão Matriz V7 -> Bitboard
            mat = np.frombuffer(estado_bytes, dtype=np.int8).reshape(9, 7)
            edges = 0
            idx = 0
            for r in range(9):
                for c in range(7):
                    if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                        if mat[r, c] == 9: edges |= (1 << idx)
                        idx += 1

            # A TT deve ser limpa POR TABULEIRO (Compartilhada entre os 31 traços iniciais)
            tt = {} 
            scores = np.full(31, -1_000_000_000.0, dtype=np.float32)
            
            for i in range(31):
                if not (edges & (1 << i)):
                    new_e = edges | (1 << i)
                    closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
                    
                    res = (closed + solve_minimax_bitboard(new_e, 6, -10001, 10001, True, tt)) if closed > 0 else \
                          solve_minimax_bitboard(new_e, 6, -10001, 10001, False, tt)
                    scores[i] = float(res)

            # Definição do Argmax (Melhor Jogada)
            validos = [i for i, s in enumerate(scores) if s > -1e8]
            if not validos:
                melhor_rotulo = ""
            else:
                max_s = np.max(scores[validos])
                best_idx = random.choice([i for i in validos if scores[i] == max_s])
                melhor_rotulo = bit_to_label[best_idx]
                
            resultados.append({
                "estado_bytes": estado_bytes,
                "melhor_jogada": melhor_rotulo,
                "scores": scores.tolist()
            })
        yield pd.DataFrame(resultados)

In [0]:
spark.conf.set('spark.databricks.execution.timeout', 86400)

In [0]:
# Orquestração Principal com Checkpoints e Gravação Atômica

# Mapeia todos os NPZs existentes e separa os pendentes
todos_npzs = sorted(INPUT_DIR.glob("dataset_pequeno_*.npz"))
pendentes = []

print(f"Verificando {len(todos_npzs)} arquivos para montar a fila de trabalho...")
for path in todos_npzs:
    if ".tmp." in path.name: continue # Ignora resquícios de arquivos temporários corrompidos
    with np.load(path) as d:
        # Se o primeiro estado do NPZ está vazio, o arquivo inteiro é considerado pendente
        if d["melhor_jogada"].size > 0 and d["melhor_jogada"][0] != "":
            continue 
        pendentes.append(path)

print(f"Total de arquivos já processados: {len(todos_npzs) - len(pendentes)}")
print(f"Total de arquivos na fila: {len(pendentes)}\n")

# Processa a fila em Lotes de Segurança (Checkpoints)
schema_spark = "estado_bytes BINARY, melhor_jogada STRING, scores ARRAY<FLOAT>"

for i in range(0, len(pendentes), LOTE_ARQUIVOS):
    lote_atual_paths = pendentes[i : i + LOTE_ARQUIVOS]
    estados_unicos_bytes = set()
    
    # 1. Extrai apenas os estados únicos deste LOTE
    for path in lote_atual_paths:
        with np.load(path) as d:
            for est in d["estados"]:
                estados_unicos_bytes.add(est.tobytes())
                
    print(f"--- Iniciando Lote {i // LOTE_ARQUIVOS + 1} ({len(lote_atual_paths)} arquivos) ---")
    print(f"Enviando {len(estados_unicos_bytes)} estados únicos para o cluster Spark...")
    
    start_time = time.time()
    
    # 2. Execução Distribuída via Spark mapInPandas
    pdf_in = pd.DataFrame({"estado_bytes": list(estados_unicos_bytes)})
    df_spark = spark.createDataFrame(pdf_in)
    
    # Força a paralelização lidando com o modo 'auto' do Databricks Serverless
    conf_particoes = spark.conf.get("spark.sql.shuffle.partitions", "200")
    try:
        num_particoes = int(conf_particoes)
    except ValueError:
        num_particoes = 200  # Fallback seguro se a configuração retornar 'auto'
        
    df_spark = df_spark.repartition(num_particoes)
    
    df_result = df_spark.mapInPandas(processar_lote_pandas, schema=schema_spark)
    resultados_lote = df_result.collect()
    
    # Popula o cache local com as respostas
    cache_local = {
        bytes(row.estado_bytes): (row.melhor_jogada, np.array(row.scores, dtype=np.float32))
        for row in resultados_lote
    }
    
    print(f"Cálculo concluído em {(time.time() - start_time):.1f}s. Gravando no disco...")
    
    # 3. Gravação Atômica no Disco
    for path in lote_atual_paths:
        with np.load(path) as d:
            n = d["estados"].shape[0]
            mj_arr = np.empty(n, dtype="<U5")
            smj_arr = np.empty((n, 31), dtype=np.float32)
            
            for idx_arr in range(n):
                rotulo, scores = cache_local[d["estados"][idx_arr].tobytes()]
                mj_arr[idx_arr] = rotulo
                smj_arr[idx_arr] = scores
                
            tmp_path = path.with_name(path.stem + ".tmp.npz")
            np.savez_compressed(
                tmp_path,
                estados=d["estados"],
                qtd_tracos=d["qtd_tracos"],
                score_jogada=d["score_jogada"],
                depth_jogada=d["depth_jogada"],
                depth_geracao=d["depth_geracao"],
                melhor_jogada=mj_arr,
                score_melhor_jogada=smj_arr,
                depth_melhor_jogada=np.full(n, DEPTH_TARGET, dtype=np.int8),
                labels_canonicos=d["labels_canonicos"]
            )
            os.replace(tmp_path, path) # Substituição Atômica Segura
        
        print(f"  -> {path.name} gravado e seguro.")
        
print("\nTodo o trabalho foi finalizado com sucesso!")

Verificando 152 arquivos para montar a fila de trabalho...
Total de arquivos já processados: 0
Total de arquivos na fila: 152

--- Iniciando Lote 1 (4 arquivos) ---
Enviando 16798 estados únicos para o cluster Spark...
Cálculo concluído em 559.7s. Gravando no disco...
  -> dataset_pequeno_0001.npz gravado e seguro.
  -> dataset_pequeno_0002.npz gravado e seguro.
  -> dataset_pequeno_0003.npz gravado e seguro.
  -> dataset_pequeno_0004.npz gravado e seguro.
--- Iniciando Lote 2 (4 arquivos) ---
Enviando 16851 estados únicos para o cluster Spark...
Cálculo concluído em 536.8s. Gravando no disco...
  -> dataset_pequeno_0005.npz gravado e seguro.
  -> dataset_pequeno_0006.npz gravado e seguro.
  -> dataset_pequeno_0007.npz gravado e seguro.
  -> dataset_pequeno_0008.npz gravado e seguro.
--- Iniciando Lote 3 (4 arquivos) ---
Enviando 16787 estados únicos para o cluster Spark...
Cálculo concluído em 536.9s. Gravando no disco...
  -> dataset_pequeno_0009.npz gravado e seguro.
  -> dataset_pe